In [ ]:
#binary Stacking
import pandas as pd
import numpy as np
import os, time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# =============================
# Load and Merge All CSVs
# =============================
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)
        if 'type' in df.columns:
            df = df.drop(columns=['type'])
        all_data.append(df)

df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

y = df['label']
X = df.drop(columns=['label'])

print("\nClass distribution in dataset:")
print(y.value_counts())

# =============================
# Train-test split (80/20)
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning Manual Stacked Model (GBT + XGB + LGBM)...\n")

# =============================
# Define Base Models
# =============================
gbt = GradientBoostingClassifier(
    n_estimators=3200,
    learning_rate=0.05,
    loss='log_loss',
    random_state=42
)
xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    n_estimators=1200,       # more boosting rounds
    learning_rate=0.05,      # smaller LR for stability
    max_depth=7,             # deeper trees
    min_child_weight=5,      # regularization to reduce overfitting
    subsample=0.8,           # row sampling
    colsample_bytree=0.8,    # feature sampling
    gamma=0.2,               # min loss reduction (regularization)
    reg_lambda=1.0,          # L2 regularization
    reg_alpha=0.5,           # L1 regularization
    scale_pos_weight=1,      # adjust if dataset is imbalanced
    random_state=42,
    n_jobs=-1
)
lgbm = LGBMClassifier(random_state=42)

meta_learner = LogisticRegression(max_iter=2000, random_state=42)

# =============================
# Train Base Models
# =============================
start_time = time.time()
gbt.fit(X_train, y_train)
xgb.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

# =============================
# Create Meta Features
# =============================
meta_X_train = np.column_stack([
    gbt.predict_proba(X_train)[:, 1],
    xgb.predict_proba(X_train)[:, 1],
    lgbm.predict_proba(X_train)[:, 1]
])

meta_X_test = np.column_stack([
    gbt.predict_proba(X_test)[:, 1],
    xgb.predict_proba(X_test)[:, 1],
    lgbm.predict_proba(X_test)[:, 1]
])

# =============================
# Train Meta-Learner
# =============================
meta_learner.fit(meta_X_train, y_train)
train_time = time.time() - start_time

# =============================
# Predictions & Metrics
# =============================
y_pred = meta_learner.predict(meta_X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)

# =============================
# Results
# =============================
print("\n===== STACKING RESULTS (GBT + XGB + LGBM) =====")
print(f"Training Time: {train_time:.2f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print("Confusion Matrix:")
print(cm)


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N

C:\Users\60105\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:183: UserWarning: [18:42:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 17234112, number of negative: 637104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.434700 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1426
[LightGBM] [Info] Number of data points in the train set: 17871216, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.964350 -> initscore=3.297713
[LightGBM] [Info] Start training from score 3.297713

===== STACKING RESULTS (GBT + XGB + LGBM) =====
Training Time: 75321.83 seconds
Accuracy: 0.9957
F1 Score: 0.9978
Recall: 0.9993
Precision: 0.9962
False Positive Rate (FPR): 0.1037
Confusion Matrix:
[[ 142756   16520]
 [   2841 4305688]]
